# ema-first-moment composite — cx19: Adam dual m & v EMA buffers in one pass

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `ema-first-moment`, `ema-second-moment`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "ema-first-moment"
DD_ATOM_IDS = ["ema-first-moment", "ema-second-moment"]
DD_SUBTOPICS = ["Optimizer: Adam EMA first moment", "Optimizer: Adam EMA second moment"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## How these two atoms compose

Adam tracks TWO running buffers per parameter:
1. **First moment `m`** — an EMA of the raw gradient `g`. Acts as a momentum-like term: smoothes the descent direction.
2. **Second moment `v`** — an EMA of the SQUARED gradient `g**2`. Acts as a per-coordinate adaptive scale: directions with large historical |g| get smaller effective steps.

The two recurrences are INDEPENDENT (no cross-term):
```python
m = beta1 * m + (1 - beta1) * g
v = beta2 * v + (1 - beta2) * g.pow(2)
```
Both share the same shape as the parameter (and the gradient). Real `torch.optim.Adam` runs both updates inside ONE per-parameter loop so the `(m, v, g)` triple is in cache together — that's the canonical composition.

**Why both atoms together.** Either alone is just a moving average; together they ARE Adam's state. Bias correction and the denominator wiring come later — this drill is purely the two buffer recurrences.

**`.copy_()` not assignment.** `m = beta1 * m + ...` rebinds the local name; the caller's list still points to the old tensor. `m.copy_(...)` mutates the storage in place so the caller sees the update.

### Composite Exercise — Adam dual m & v EMA buffers in one pass

**Atoms exercised together**: `ema-first-moment`, `ema-second-moment`

Implement `cx19_update_m_and_v(m_list, v_list, grad_list, beta1, beta2)`.

For each triple `(m, v, g)` from the three same-length lists, update IN PLACE:
1. `m <- beta1 * m + (1 - beta1) * g`
2. `v <- beta2 * v + (1 - beta2) * g.pow(2)`

Use `m.copy_(...)` and `v.copy_(...)` so the original tensor storage (its `data_ptr()`) survives the call. Return `None`.

The test checks: (a) both buffers update from a zero start by the correct closed-form; (b) `v` stays non-negative even with negative gradients (since we square); (c) the i-th `(m_i, v_i)` is independent of the j-th (no cross-talk between params); (d) after many steps of constant `g`, `m` converges to `g` and `v` converges to `g**2`; (e) tensor `data_ptr()` is preserved across many calls.

In [ ]:
def cx19_update_m_and_v(m_list, v_list, grad_list, beta1, beta2):
    for m, v, g in zip(m_list, v_list, grad_list):
        # Atom A (ema-first-moment): EMA of g.
        m.copy_(beta1 * m + (1.0 - beta1) * g)
        # Atom B (ema-second-moment): EMA of g**2.
        v.copy_(beta2 * v + (1.0 - beta2) * g.pow(2))


<details><summary>Show solution — cx19</summary>

```python
def cx19_update_m_and_v(m_list, v_list, grad_list, beta1, beta2):
    for m, v, g in zip(m_list, v_list, grad_list):
        # Atom A (ema-first-moment): EMA of g.
        m.copy_(beta1 * m + (1.0 - beta1) * g)
        # Atom B (ema-second-moment): EMA of g**2.
        v.copy_(beta2 * v + (1.0 - beta2) * g.pow(2))
```

**Why one combined loop.** Both updates are independent — `m` reads only m & g, `v` reads only v & g. Putting them in the same per-param loop is purely a cache / memory locality win: the `(m, v, g)` triple is touched once per param.

**Order is free.** You could update `v` first then `m` — the result is the same. PyTorch's `torch.optim.Adam` updates `exp_avg` first then `exp_avg_sq` by convention.
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx19'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx19',
        'subtopics': ["Optimizer: Adam EMA first moment", "Optimizer: Adam EMA second moment"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()